# Train Phase 2-beta v4 (silver + CoT)Trains a fresh Qwen-1.5B + LoRA r=16 adapter on 5 gold corpora plus the 2,123 LIARArg silver records (with the teacher's Chain of Thought traces preserved in the training target). This is the headline model of the paper.Config: `configs/phase2_beta_qwen1.5b_lora_v4.yaml`. Requires the silver labels to be present at `phase2_data/silver/liararg_train_silver.jsonl`.About 29 hours on a single GTX 1080 Ti.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
git pull origin main

# Symlink the silver file with the right naming pattern
# (load_unified_corpus parses filenames as <source>_<split>.jsonl,
# so 'liararg_silver_train.jsonl' becomes source='liararg_silver', split='train')
ln -sf ../silver/liararg_train_silver.jsonl \
       phase2_data/unified/liararg_silver_train.jsonl

echo "=== unified/ now contains ==="
ls -la phase2_data/unified/ | grep -E '\.jsonl|silver'

echo ""
echo "=== verify loader picks up silver ==="
python3 -c "
import sys; sys.path.insert(0, '.')
from src.phase2.dataset import load_unified_corpus
records = load_unified_corpus('phase2_data/unified', exclude_sources=('liararg',))
from collections import Counter
by_source = Counter(r['source_dataset'] for r in records)
by_split = Counter(r['split'] for r in records)
by_label = Counter(r.get('label_kind', 'gold') for r in records)
print(f'Total: {len(records)} records')
print(f'By source: {dict(by_source)}')
print(f'By split:  {dict(by_split)}')
print(f'By label_kind: {dict(by_label)}')
"

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p eval_logs

nohup python3 -u -m scripts.phase2.train_phase2_beta \
    --config configs/phase2_beta_qwen1.5b_lora_v4.yaml \
    --exclude liararg \
    > eval_logs/phase2beta_v4_train.log 2>&1 &
echo "Train PID: $!"
sleep 15
tail -25 eval_logs/phase2beta_v4_train.log